> **Adaptación de CAIS, Intro to ML Safety.** Este notebook adapta al español el notebook L6 del curso *Intro to ML Safety* (course.mlsafety.org) del Center for AI Safety. Reproduced with permission of Center for AI Safety, Inc. CAIS no respalda este curso ni tiene relación con él.
>
> Métodos: FGSM (Goodfellow, Shlens y Szegedy, 2014, arXiv:1412.6572) y PGD (Madry et al., 2017, arXiv:1706.06083). Los requisitos instalan la biblioteca `robustness` de MadryLab (MIT) como dependencia; no se copia su código.
>
> Avisos y licencias completos: `NOTICE.md` en la raíz del repositorio.

# Robustez adversarial

Los ejemplos adversariales son entradas diseñadas para que un sistema de aprendizaje automático falle. En este notebook implementamos los dos ataques clásicos, FGSM y PGD, sobre una red ya entrenada con ImageNet, los probamos contra un modelo normal y contra uno entrenado de forma adversarial, y medimos cómo cae la exactitud según el número de pasos del ataque.

> **Nota de la adaptación (curso de seguridad de la IA, lección m04/l01).** Es la adaptación al español del notebook de la lección L6 del curso *Intro to ML Safety*. El código es el del original; se han traducido los textos y los comentarios. Es un **esqueleto de ejercicio**: las celdas con "Escribe aquí tu código" están vacías a propósito y hay que rellenarlas. Necesita una GPU con PyTorch y el material del repositorio del ejercicio (véanse los requisitos). No se ha ejecutado al prepararlo.

## Requisitos para ejecutar en local

El original corría en Google Colab: clonaba un repositorio de GitHub e instalaba dos paquetes con versiones fijadas. Aquí se ejecuta en el equipo propio con GPU. Antes de empezar:

1. Un entorno de Python con Jupyter, `numpy`, `matplotlib`, PyTorch con CUDA (con la orden que da su web para tu versión de CUDA) y `torchvision`.
2. El material del ejercicio. El original hacía esto, que ahora debes hacer tú a mano en una terminal, junto a este notebook (necesita conexión):

```bash
git clone --branch adversarial https://github.com/oliverzhang42/safety.git
pip install robustness==1.2.1.post2
pip install torchvision==0.10.0
```

   El repositorio `safety` aporta los módulos `safety.utils` y `safety.lesson1.adversarial`, un lote de imágenes de ImageNet con sus etiquetas y el modelo con entrenamiento adversarial.
3. **Aviso.** Las versiones fijadas (`robustness==1.2.1.post2` y `torchvision==0.10.0`) son de 2021 y es probable que no se instalen con Python 3.13, ni que sean compatibles con un PyTorch reciente (sin verificar). Conviene crear un entorno aparte con las versiones que pedía el original y dejar que él fije las demás.
4. Abre el notebook desde la carpeta que contiene `safety/`. La celda siguiente lo comprueba.

In [ ]:
# Requisitos para ejecución local (sustituye a las celdas de Colab que clonaban el
# repositorio con !git clone e instalaban paquetes con !pip3 install).
import importlib.util
import os

if importlib.util.find_spec("safety") is None:
    raise FileNotFoundError(
        "No se encuentra la carpeta 'safety/'. Clona el repositorio del ejercicio (rama 'adversarial') "
        "junto a este notebook y ábrelo desde esa carpeta. Carpeta actual: " + os.getcwd()
    )

import torch
if not torch.cuda.is_available():
    print("Aviso: PyTorch no ve ninguna GPU. Las celdas usan .cuda() y fallarán sin ella.")
print("Material del ejercicio encontrado en:", os.getcwd())

## Introducción y preparación

Los ejemplos adversariales son ejemplos diseñados para hacer que un sistema de aprendizaje automático funcione mal. Aquí, un adversario toma una imagen real de un panda y le suma un ruido generado de forma adversarial para obtener el ejemplo adversarial. El ruido está diseñado para quedar a poca distancia de la imagen original, de modo que a una persona le sigue pareciendo un panda. Sin embargo, el modelo pasa a creer que es un gibón con un 99,3 % de confianza.

(El original mostraba aquí una imagen alojada en Google Drive; se ha quitado de la adaptación.)

In [ ]:
# Importamos todas las bibliotecas necesarias

import torch
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from safety.utils import utils
from safety.lesson1 import adversarial
from torch import nn
from torchvision import models

%matplotlib inline

## Primer ataque adversarial con FGSM

### FGSM no dirigido

El primer método que vemos es el método rápido del signo del gradiente (*Fast Gradient Sign Method*, FGSM) no dirigido, propuesto por [Goodfellow et al.](https://arxiv.org/pdf/1412.6572.pdf). El ataque construye los ejemplos adversariales así:

$$x_\text{adv} = x + \epsilon\cdot\text{sign}(\nabla_xJ(\theta, x, y))$$

donde

*   $x_\text{adv}$ : imagen adversarial.
*   $x$ : imagen de entrada original.
*   $y$ : etiqueta original de la entrada.
*   $\epsilon$ : multiplicador que garantiza que las perturbaciones sean pequeñas.
*   $\theta$ : parámetros del modelo.
*   $J$ : pérdida.

La formulación actual se considera "no dirigida" porque solo busca maximizar la pérdida, en lugar de engañar al modelo para que prediga una etiqueta concreta.

¡Prueba a implementar tú mismo el método FGSM no dirigido para un lote de imágenes!

In [ ]:
def untargeted_FGSM(x_batch, true_labels, network, normalize, eps=8/255., **kwargs):
  """Genera un lote de ejemplos adversariales FGSM no dirigidos

  Args:
    x_batch (torch.Tensor): el lote de ejemplos de entrada sin normalizar.
    true_labels (torch.Tensor): el lote de etiquetas verdaderas de los ejemplos.
    network (nn.Module): la red que se ataca.
    normalize (function): una función que normaliza un lote de imágenes
        según la normalización estándar de ImageNet.
    eps (float): la cota de las perturbaciones.
  """
  loss_fn = nn.CrossEntropyLoss(reduce="mean")
  x_batch.requires_grad = True

  #############################
  # ¡Escribe aquí tu código!  #
  #############################

  return x_adv

In [ ]:
# Prueba el método
adversarial.test_untargeted_attack(untargeted_FGSM, eps=8/255.)

Si todo va bien, el modelo debería pasar de predecir "panda gigante" a predecir "oso pardo" u otra clase. Prueba también a aumentar el épsilon para ver el ruido con más claridad.

### FGSM dirigido

Además del FGSM no dirigido, que simplemente busca maximizar la pérdida, también podemos crear ataques adversariales dirigidos. Lo hacemos con la ecuación siguiente:

$$x_{adv} = x - \epsilon\cdot\text{sign}(\nabla_xJ(\theta, x, y_{target}))$$

donde

*   $x_{adv}$ : imagen adversarial.
*   $x$ : imagen de entrada original.
*   $y_{target}$ : la etiqueta objetivo.
*   $\epsilon$ : multiplicador que garantiza que las perturbaciones sean pequeñas.
*   $\theta$ : parámetros del modelo.
*   $J$ : pérdida.

¡Prueba a implementar tú mismo el método FGSM dirigido para un lote de imágenes!

In [ ]:
def targeted_FGSM(x_batch, target_labels, network, normalize, eps=8/255., **kwargs):
  """Genera un lote de ejemplos adversariales FGSM dirigidos

  Args:
    x_batch (torch.Tensor): el ejemplo de entrada sin normalizar.
    target_labels (torch.Tensor): las etiquetas que el modelo predecirá tras el ataque.
    network (nn.Module): la red que se ataca.
    normalize (function): una función que normaliza un lote de imágenes
        según la normalización estándar de ImageNet.
    eps (float): la cota de las perturbaciones.
  """
  loss_fn = nn.CrossEntropyLoss(reduce="mean")
  x_batch.requires_grad = True

  #############################
  # ¡Escribe aquí tu código!  #
  #############################

  return x_adv

**Ten en cuenta que, aunque la implementación sea perfecta, FGSM no consigue generar ataques dirigidos eficaces, así que no esperes que la imagen de salida asigne una probabilidad alta a la etiqueta objetivo.**

In [ ]:
adversarial.test_targeted_attack(targeted_FGSM, target_idx=8, eps=8/255.)

## Otros ataques adversariales

## Implementar la normalización L2 y el recorte

Implementaremos unas funciones auxiliares que usaremos para el método de descenso de gradiente proyectado (*Projected Gradient Descent*, PGD) en norma L2 que aparece más abajo.

La función `normalize_l2` devolverá el valor siguiente:

$$\frac{x}{||x||_{2}}$$

La función `tensor_clamp_l2` calculará y devolverá este valor:

\begin{equation}
    X=
    \begin{cases}
      clamp(x), & \text{si}\ ||x-c||_2 > r \\
      x, & \text{en otro caso}
    \end{cases}
  \end{equation}

donde $ \text{clamp}(x) = c + \frac{x-c}{||x-c||_2} ⋅ r$, X es el valor devuelto, x es la entrada, c (centro) es un tensor de la misma forma que x y r (radio) es un valor escalar.

¡Prueba a implementar tú mismo las versiones por lotes de `normalize_l2` y `tensor_clamp_l2`!

In [ ]:
def normalize_l2(x_batch):
    """
    Espera x_batch.shape == [N, C, H, W]
    donde N es el tamaño del lote,
    C son los canales (o colores en nuestro caso),
    H, W son el alto y el ancho respectivamente.

    Nota: para calcular la norma L2 de una imagen conviene aplanar sus dimensiones (con cuidado de conservar la dimensión del lote de x_batch).
    """
    #############################
    # ¡Escribe aquí tu código!  #
    #############################

In [ ]:
def tensor_clamp_l2(x_batch, center, radius):
    """Recorte por lotes de x dentro de la bola L2 alrededor de center con el radio dado."""

    #############################
    # ¡Escribe aquí tu código!  #
    #############################

In [ ]:
def PGD_l2(x_batch, true_labels, network, normalize, num_steps=20, step_size=3./255, eps=128/255., **kwargs):
        """
        :return: lote de imágenes perturbadas
        """
        # Inicializamos nuestra imagen adversarial
        x_adv = x_batch.detach().clone()
        x_adv += torch.zeros_like(x_adv).uniform_(-eps, eps)

        for _ in range(num_steps):
            x_adv.requires_grad_()

            # Calculamos los gradientes
            with torch.enable_grad():
              logits = network(normalize(x_adv))
              loss = F.cross_entropy(logits, true_labels, reduction='sum')

            # Normalizamos los gradientes con tu L2
            grad = normalize_l2(torch.autograd.grad(loss, x_adv, only_inputs=True)[0])

            # Damos un paso en la dirección del gradiente.
            x_adv = x_adv.detach() + step_size * grad
            # Proyectamos (recortando) la imagen adversarial de vuelta a la hiperesfera
            # alrededor de la imagen.
            x_adv = tensor_clamp_l2(x_adv, x_batch, eps).clamp(0, 1)

        return x_adv

Prueba las funciones auxiliares que has escrito. Fíjate en cómo los hiperparámetros cambian según el ataque que se use. Más abajo hay más ejemplos.

In [ ]:
adversarial.test_untargeted_attack(PGD_l2, eps=128/255.)

Además de tus implementaciones de FGSM, te damos una implementación del descenso de gradiente proyectado (PGD) de [Madry et al.](https://arxiv.org/pdf/1706.06083.pdf). Como se explicó en la lección, PGD puede verse como una versión más fuerte de FGSM que aplica FGSM muchas veces. Damos tanto la versión dirigida como la no dirigida.

In [ ]:
def untargeted_PGD(x_batch, true_labels, network, normalize, num_steps=10, step_size=0.01, eps=8/255., **kwargs):
  """Genera un lote de ejemplos adversariales PGD no dirigidos

  Args:
    x_batch (torch.Tensor): el lote de ejemplos de entrada sin normalizar.
    true_labels (torch.Tensor): el lote de etiquetas verdaderas de los ejemplos.
    network (nn.Module): la red que se ataca.
    normalize (function): una función que normaliza un lote de imágenes
        según la normalización estándar de ImageNet.
    num_steps (int): el número de pasos de PGD.
    step_size (float): el tamaño de cada paso de PGD.
    eps (float): la cota de las perturbaciones.
  """
  x_adv = x_batch.detach().clone()
  x_adv += torch.zeros_like(x_adv).uniform_(-eps, eps)

  for i in range(num_steps):
    x_adv.requires_grad_()

    # Calculamos los gradientes
    with torch.enable_grad():
      logits = network(normalize(x_adv))
      loss = F.cross_entropy(logits, true_labels, reduction='sum')
    grad = torch.autograd.grad(loss, x_adv, only_inputs=True)[0]

    # Damos un paso de gradiente
    x_adv = x_adv.detach() + step_size * torch.sign(grad.detach())

    # Proyectamos la imagen a la bola.
    x_adv = torch.maximum(x_adv, x_batch - eps)
    x_adv = torch.minimum(x_adv, x_batch + eps)

  return x_adv

In [ ]:
adversarial.test_untargeted_attack(untargeted_PGD, eps=8/255.)

In [ ]:
def targeted_PGD(x_batch, target_labels, network, normalize, num_steps=100, step_size=0.01, eps=8/255., **kwargs):
  """Genera un lote de ejemplos adversariales PGD dirigidos

  Args:
    x_batch (torch.Tensor): el lote de ejemplos de entrada preprocesados.
    target_labels (torch.Tensor): las etiquetas que el modelo predecirá tras el ataque.
    network (nn.Module): la red que se ataca.
    normalize (function): una función que normaliza un lote de imágenes
        según la normalización estándar de ImageNet.
    num_steps (int): el número de pasos de PGD.
    step_size (float): el tamaño de cada paso de PGD.
    eps (float): la cota de las perturbaciones.
  """
  x_adv = x_batch.detach().clone()
  x_adv += torch.zeros_like(x_adv).uniform_(-eps, eps)

  for i in range(num_steps):
    x_adv.requires_grad_()

    # Calculamos los gradientes
    with torch.enable_grad():
      logits = network(normalize(x_adv))
      loss = F.cross_entropy(logits, target_labels, reduction='sum')
    grad = torch.autograd.grad(loss, x_adv, only_inputs=True)[0]

    # Damos un paso de gradiente
    # Ten en cuenta que esta vez usamos descenso de gradiente en lugar de ascenso
    x_adv = x_adv.detach() - step_size * torch.sign(grad.detach())

    # Proyectamos la imagen a la bola
    x_adv = torch.maximum(x_adv, x_batch - eps)
    x_adv = torch.minimum(x_adv, x_batch + eps)

  return x_adv

In [ ]:
# ¡Prueba a cambiar target_idx!
adversarial.test_targeted_attack(targeted_PGD, target_idx=1, eps=8/255.)

## Ataques a modelos entrenados de forma adversarial
Dedicamos esta sección a atacar un modelo entrenado de forma adversarial. Recuerda que un modelo "entrenado de forma adversarial" ha visto durante el entrenamiento una gran cantidad de ejemplos adversariales y se ha entrenado específicamente para reconocerlos bien.

En esta sección queremos mostrar que los ataques adversariales tienen un aspecto muy distinto cuando se dirigen contra un modelo entrenado de forma adversarial.

El modelo que usamos procede de este repositorio: es una ResNet18 robusta en L∞ entrenada con ejemplos adversariales de ϵ=8/255.

### Atacar modelos entrenados normalmente

In [ ]:
# Atacamos un modelo normal (solo se admiten métodos dirigidos)
adversarial.attack_normal_model(
    targeted_PGD,
    target_idx=10,
    eps=8/255.,
    num_steps=10,
    step_size=0.01
)

### Atacar modelos entrenados de forma adversarial

In [ ]:
# Atacamos un modelo entrenado de forma adversarial (solo se admiten métodos dirigidos)
adversarial.attack_adversarially_trained_model(
    targeted_PGD,
    target_idx=10,
    eps=8/255.,
    num_steps=10,
    step_size=0.01
)

### Comparar ataques adversariales contra distintos modelos

Dedica unos minutos a jugar con el código anterior. Apunta tres observaciones sobre en qué se diferencia atacar un modelo entrenado de forma adversarial de atacar uno normal.

Ejemplo de respuestas:

1. La confianza de los modelos típicos es mayor que la de los entrenados de forma adversarial
2. [Escribe una observación]
3. [Escribe una observación]

## Exactitud frente al número de pasos de PGD

En esta sección queremos ver cómo varía la exactitud al cambiar el número de pasos de PGD. Tu primera tarea es escribir una función que calcule la exactitud del modelo sobre imágenes generadas de forma adversarial. Para este caso usamos PGD no dirigido.

In [ ]:
def adversarial_accuracy(x_batch, true_labels, network, normalize, num_steps=10, step_size=0.01, eps=8/255.):
    """Genera un lote de ejemplos adversariales con `untargeted_PGD`. Después
    calcula y devuelve la exactitud sobre esos ejemplos.

    Args:
      x_batch (torch.Tensor): el lote de ejemplos de entrada preprocesados.
      true_labels (torch.Tensor): el lote de etiquetas verdaderas de los ejemplos.
      network (nn.Module): la red que se ataca.
      normalize (function): una función que normaliza un lote de imágenes
          según la normalización estándar de ImageNet.
      num_steps (int): el número de pasos de PGD.
      step_size (float): el tamaño de cada paso de PGD.
      eps (float): la cota de las perturbaciones.

    Nota: considera que la predicción de la red es la clase con la salida (logit) más alta.
    """
    x_adv = untargeted_PGD(x_batch, true_labels, network, normalize, num_steps=num_steps, step_size=step_size, eps=eps)

    #############################
    # ¡Escribe aquí tu código!  #
    #############################

    return accuracy

Después, usa la función anterior para dibujar la exactitud frente al número de pasos de PGD.

In [ ]:
x_batch = torch.load('safety/lesson1/imagenet_batch').cuda()
true_labels = torch.load('safety/lesson1/imagenet_labels').cuda()
network = adversarial.get_adv_trained_model().eval()
normalization_function = utils.IMAGENET_NORMALIZE

#############################
# ¡Escribe aquí tu código!  #
#############################

# llama a adversarial_accuracy(...) con varios valores de num_steps y dibuja la gráfica